# 集群平台、网络存储与可靠性补充线 · 第 3/8 课：存储层级、数据集与 Checkpoint 带宽

> 状态：**参考答案版**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：计算 checkpoint 的有效写入时间和带宽需求，区分本地 NVMe、共享文件系统与对象存储角色。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`runtime/lesson06` 讲进程内输入流水；本课关注集群存储层、元数据压力、checkpoint 突发和恢复路径。

前置：Linux/网络基础、runtime 补充线、train 分布式章节。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

数据通常从对象/并行文件系统进入节点缓存/NVMe，再由 DataLoader 消费；checkpoint 可先本地落盘再异步上传。GDS 可在兼容路径减少 CPU bounce buffer。

### 数据与控制如何流动

数据读取沿持久存储→节点缓存/NVMe→进程/GPU 流动；checkpoint 反向生成各 rank shard，先写临时对象，校验齐全后发布 manifest/commit 标记，再异步复制并按保留策略回收旧版本。

### 正确性条件与常见误区

checkpoint 只有在 manifest、所有 shards 和提交标记一致后才可恢复；带宽估算需包含副本/纠删码、压缩和并发作业。大量小文件会压垮元数据服务。

### 性能、成本与工程取舍

本地 NVMe 快但节点故障会丢；共享存储可靠/可发现但有争用；对象存储容量大而单对象延迟、列表一致性和 API 语义需设计。

## 具体演示

总 checkpoint 4 TB，每 30 分钟写完且后端写放大 1.5：平均需 3.33 GB/s；若 100 作业同时触发，瞬时需求不可简单按单作业规划。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐 checkpoint 写入时间与周期内最低平均带宽。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def checkpoint_io(checkpoint_bytes, backend_Bps, write_amplification, interval_s):
    if min(checkpoint_bytes, backend_Bps, write_amplification, interval_s) <= 0:
        raise ValueError("all values must be positive")
    physical_bytes = checkpoint_bytes * write_amplification
    duration = physical_bytes / backend_Bps
    # TODO：周期内摊销的最低平均后端带宽。
    required_average = ______
    return duration, required_average

duration, avg = checkpoint_io(4e12, 10e9, 1.5, 1800)
assert duration == 600 and abs(avg - 3.3333333333333335e9) < 1


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么“所有 rank 各写一个小文件”在规模扩大后常失败？

**你的答案：**


### Q2

本地 NVMe checkpoint 完成后即可向训练宣称“可恢复”吗？

**你的答案：**


### Q3

GDS 能减少 CPU copy，为什么未必提升小样本随机读取？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考答案（仅 answer 分支）

先独立完成。核对后请改变一个规模或故障条件重新推演。

In [ ]:
def checkpoint_io(checkpoint_bytes, backend_Bps, write_amplification, interval_s):
    if min(checkpoint_bytes, backend_Bps, write_amplification, interval_s) <= 0:
        raise ValueError("all values must be positive")
    physical_bytes = checkpoint_bytes * write_amplification
    duration = physical_bytes / backend_Bps
    required_average = physical_bytes / interval_s
    return duration, required_average

duration, avg = checkpoint_io(4e12, 10e9, 1.5, 1800)
assert duration == 600 and abs(avg - 3.3333333333333335e9) < 1


### Q1 参考答案

文件数、open/close、目录锁和 metadata RPC 按 rank 增长，带宽尚未饱和前 metadata server 已成为瓶颈。应聚合、使用分片格式/对象并控制并发。

### Q2 参考答案

若恢复要求节点故障后仍可用，则不行。需异步复制到持久后端并原子发布 manifest/commit；在此之前只能恢复进程故障而不能恢复节点盘丢失。

### Q3 参考答案

小 I/O 受文件系统、元数据、提交和设备延迟主导，注册/提交开销也占比高。GDS 更适合足够大、对齐、可并行的数据路径，仍需 workload benchmark。

## 参考资料

- [GPUDirect Storage](https://docs.nvidia.com/gpudirect-storage/)
- [PyTorch Distributed Checkpoint](https://docs.pytorch.org/docs/stable/distributed.checkpoint.html)
- [torch.utils.data](https://docs.pytorch.org/docs/stable/data.html)

API 与平台能力会演进；部署前应按目标版本重新核对。